|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 4:</h2>|<h1>The Scheduler<h1>|
|<h2>Section:</h2>|<h1>Chunked prefill<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: one token budget, two kinds of work<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(0)

Write the mixed-batch scheduler.

One token budget per step. Decoding sequences contribute one token each;
prefilling ones contribute a slice of their prompt. No step is allowed to
become a long prefill that stalls everybody.

This is stage 11, and it is the last purely logical scheduler in the course.

In [ ]:
### run this cell

# 32 short conversations that arrive at once and settle into decoding,
# then at step 40 somebody pastes in 4096 tokens.
#   [arrival_step, prompt_len, output_len]
reqs = [[0, 64, 400] for _ in range(32)] + [[40, 4096, 100]]
print(f'{len(reqs)} requests; the big one arrives at step {reqs[-1][0]}')

# Exercise 1: the budget loop

Every step spends a fixed number of tokens. Decodes first, then as much of
the waiting prompts as still fits.

In [ ]:
def schedule(requests, budget, max_running=64):
  """requests: [[arrival_step, prompt_len, output_len], ...].
  Returns an array of (tokens_in_step, sequences_decoding_that_step).

  One budget per step. A decoding sequence contributes 1 token. A
  prefilling one contributes as much of its prompt as still fits."""
  pending = sorted(requests)
  waiting, prefilling, decoding = [], [], []
  steps, t = [], 0

  while pending or waiting or prefilling or decoding:
    while pending and pending[0][0] <= t:
      _, p, o = pending.pop(0); waiting.append([p, o])
    while waiting and len(prefilling) + len(decoding) < max_running:
      prefilling.append(waiting.pop(0))

    used = 0
    n_decoding = len(decoding)      # how many users are waiting on this step
    # decodes first. Why first? They are one token each, and a user
    # waiting on a continuation notices sooner than one waiting to start.
    for d in list(decoding):
      

    # then spend whatever budget is left on prompt chunks
    for pr in list(prefilling):
      
      # a prompt that finishes becomes a decoding sequence
      

    steps.append((used, n_decoding)); t += 1
    if used == 0 and not pending: break
  return np.array(steps)

s = schedule(reqs, budget=512)
print(f'{len(s)} steps, mean {s[:,0].mean():.0f} tokens, max {s[:,0].max()} tokens')

# Exercise 2: put a clock on it

Step cost is not proportional to tokens: measured in `part4_chk_theStall`, a
step is nearly free up to a couple of hundred tokens and linear after that.
Interpolate the measurements rather than assuming.

In [ ]:
# measured on an RTX 4080 Laptop in part4_chk_theStall. Use your own.
COST = {32: 9.4, 64: 9.9, 128: 10.0, 256: 11.4, 512: 18.7,
        1024: 35.9, 2048: 74.2, 4096: 178.7}
keys = np.array(sorted(COST))
vals = np.array([COST[k] for k in keys])

def step_cost(n):
  return float(np.interp(max(n,1), keys, vals))

print(f"{'budget':>7} {'steps':>7} {'total ms':>10} {'p99 ITL':>9} {'worst ITL':>11}")
for budget in (64, 128, 256, 512, 1024, 4096):
  s   = schedule(reqs, budget)
  ms  = 
  # a p99 over STEPS is the wrong population: one catastrophic step in
  # two hundred does not reach p99, and every user felt it. Weight each
  # step by how many sequences were decoding through it.
  itl = 
  print(f'{budget:>7} {len(s):>7} {ms.sum():>9.0f}ms {np.percentile(itl,99):>8.1f}ms {itl.max():>10.1f}ms')

# Exercise 3: the dial

Sweep the budget and plot throughput against tail latency.

In [ ]:
budgets = [64, 128, 256, 512, 1024, 2048, 4096]
tot, p99, worst = [], [], []
for b in budgets:
  s   = schedule(reqs, b)
  ms  = 
  itl = 
  tot.append(ms.sum()); p99.append(np.percentile(itl, 99)); worst.append(itl.max())

fig, ax1 = plt.subplots(figsize=(7.5,4.6))
ax1.plot(budgets, tot, 'mo-', label='total time (throughput)')
ax1.set_xscale('log', base=2)
ax1.set(xlabel='Token budget per step', ylabel='Total ms to finish everything')
ax2 = ax1.twinx()
ax2.plot(budgets, p99,   'ro--', label='p99 ITL')
ax2.plot(budgets, worst, 'r^-',  label='worst ITL')
ax2.set_yscale('log')
ax2.set_ylabel('p99 ITL (ms)')
ax1.grid(alpha=.3)
fig.legend(loc='upper center'); plt.title('The dial has two ends')
plt.tight_layout(); plt.show()

print(f'fastest overall:  budget {budgets[int(np.argmin(tot))]}')
print(f'lowest p99 ITL:   budget {budgets[int(np.argmin(p99))]}')
print(f'lowest worst ITL: budget {budgets[int(np.argmin(worst))]}')

### Before you open the solution

1. Your throughput and p99 curves go in opposite directions. Is there a
   budget that is best for both?
2. The p99 ITL stops improving below a couple of hundred tokens. Why does
   it flatten there, and which Part 1 measurement predicts that point?
3. **Look at the largest budget.** Its p99 is excellent and its worst ITL
   is terrible. Work out how many steps were catastrophic, how many users
   sat through each, and what fraction of all token-waits that is. Then
   decide which of the two numbers you would put on a dashboard.
4. You spend the budget on decodes before prefill chunks. What breaks if
   you swap the order, and who notices?